# Optimizer Benchmark Analysis

Loads CSV I/II/III and Wandb run histories to produce paper-ready plots.

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
import wandb

# ── Paths & constants ──────────────────────────────────────────────────────
os.makedirs(RESULTS_DIR, exist_ok=True)

WANDB_ENTITY  = "matanbt"
WANDB_PROJECT = "tropt-optbench"

# Paper-style aesthetics
sns.set_theme(style="whitegrid", font_scale=1.2)
PALETTE = "tab20"
FIG_DPI = 150

In [ ]:
# ── Load CSVs (needed for sections 2+) ───────────────────────────────────

RESULTS_DIR = r"C:\Users\mtnbt\My Drive (mtnbt123@gmail.com)\University\Research\Projects\2025-JailbreakToolbox\tropt\scripts\opt-bench\results"

# import sys
# project_dir = '/home/sharifm/students/matanbentov/TROPT'
# sys.path.append(project_dir)
# project_dir = '/home/sharifm/students/matanbentov/TROPT'
# os.chdir(project_dir)

# Add the project directory to the sys.path to ensure Python imports from there
# sys.path.append(project_dir)

## 1  Exp1: Optimization Dynamics

Line plot of loss over time per optimizer, for a chosen (model, message).
Shading = std across seeds. Dashed line = soft_prompt lower bound.

Only requires WandB access (no CSVs needed).

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
CHOSEN_MSG_ID = 5
# CHOSEN_MODEL  = "meta-llama/Llama-3.1-8B-Instruct"  # set to your model
CHOSEN_MODEL = "google/gemma-3-12b-it"
# CHOSEN_MODEL = "Qwen/Qwen3-8B"


api = wandb.Api()
# Filter server-side: api.runs() returns lazy runs and `run.config` comes back
# empty until wandb is forced to hydrate it. Matching on config.* forces that
# hydration, so the cfg.get(...) reads below return real values. Also covers
# both regular and soft_prompt runs in a single query.
runs = api.runs(
    f"{WANDB_ENTITY}/{WANDB_PROJECT}",
    filters={
        "state": "finished",
        "config.run_type": "optbench_whitebox",
        "config.model_name": CHOSEN_MODEL,
        "config.msg_id": CHOSEN_MSG_ID,
    },
)

# Keys to fetch from wandb (always include _runtime for time axis).
# NOTE: Do NOT include "step" — it is not logged; wandb provides "_step" automatically.
HISTORY_KEYS = ["loss", "best_loss", "total_models_stats/total_flops", "_runtime"]

# Collect per-step histories grouped by (optimizer, seed)
histories: dict[str, dict[int, pd.DataFrame]] = {}
lb_histories: dict[int, pd.DataFrame] = {}   # soft_prompt lower bound

for run in runs:
    # Belt-and-braces: if config still came back empty for any reason, force load.
    cfg = run.config
    if not cfg:
        run.load(force=True)
        cfg = run.config

    opt  = cfg.get("optimizer_name", "unknown")
    seed = cfg.get("seed", 0)
    is_soft = cfg.get("is_soft", False)

    try:
        hist = run.history(keys=HISTORY_KEYS, x_axis="_step", pandas=True)
    except Exception:
        hist = run.history(pandas=True)
    if hist.empty:
        continue
    # Normalise column names: _step -> step
    if "_step" in hist.columns:
        hist = hist.rename(columns={"_step": "step"})
    if "total_models_stats/total_flops" in hist.columns:
        hist = hist.rename(columns={"total_models_stats/total_flops": "flops"})
    if "_runtime" in hist.columns:
        hist = hist.rename(columns={"_runtime": "time"})
    if is_soft:
        lb_histories[seed] = hist
    else:
        histories.setdefault(opt, {})[seed] = hist

print(f"Loaded {sum(len(v) for v in histories.values())} runs for {len(histories)} optimizers")
if lb_histories:
    print(f"Loaded {len(lb_histories)} soft_prompt (lower-bound) runs")


In [ ]:
# ── Plot configuration ─────────────────────────────────────────────────────
X_AXIS = "flops"      # "time" (wall-clock seconds) or "flops"
Y_AXIS = "best_loss"  # "loss" (per-step) or "best_loss" (running best)

fig, ax = plt.subplots(figsize=(10, 5))

for opt_name, seed_hists in histories.items():
    color = optimizer_color(opt_name)
    # Resolve x column: prefer requested axis, fall back to step
    sample_cols = next(iter(seed_hists.values())).columns
    if X_AXIS in sample_cols:
        x_col = X_AXIS
    elif "step" in sample_cols:
        x_col = "step"
    else:
        x_col = sample_cols[0]
    # Align on common x grid by interpolating
    all_x = sorted(set(x for h in seed_hists.values() for x in h[x_col].dropna()))
    interp_losses = []
    for hist in seed_hists.values():
        valid = hist[[x_col, Y_AXIS]].dropna()
        if len(valid) < 2:
            continue
        interp_losses.append(np.interp(all_x, valid[x_col], valid[Y_AXIS]))
    if not interp_losses:
        continue
    arr = np.array(interp_losses)
    mean = arr.mean(axis=0)
    std  = arr.std(axis=0)
    ax.plot(all_x, mean, label=opt_name, color=color)
    ax.fill_between(all_x, mean - std, mean + std, alpha=0.15, color=color)

# Lower-bound dashed line
if lb_histories:
    sample_cols = next(iter(lb_histories.values())).columns
    x_col = X_AXIS if X_AXIS in sample_cols else "step"
    lb_losses = []
    all_x_lb = sorted(set(x for h in lb_histories.values() for x in h[x_col].dropna()))
    for hist in lb_histories.values():
        valid = hist[[x_col, Y_AXIS]].dropna()
        if len(valid) < 2:
            continue
        lb_losses.append(np.interp(all_x_lb, valid[x_col], valid[Y_AXIS]))
    if lb_losses:
        lb_mean = np.array(lb_losses).mean(axis=0)
        ax.plot(all_x_lb, lb_mean, linestyle="--", color="black", linewidth=1.5, label="soft_prompt (lower bound)")

X_LABELS = {"flops": "FLOPs", "time": "Wall-clock Time (s)"}
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(X_LABELS.get(X_AXIS, X_AXIS))
ax.set_ylabel(Y_AXIS.replace("_", " ").title())
ax.set_title(f"Optimization Dynamics — model: {CHOSEN_MODEL.split('/')[-1]}, msg: {CHOSEN_MSG_ID}")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)
plt.tight_layout()
fig.savefig(f"{RESULTS_DIR}/fig_dynamics_m{CHOSEN_MSG_ID}.pdf", dpi=FIG_DPI, bbox_inches="tight")
plt.show()

In [ ]:
# ── Horizontal bar chart: final best_loss per optimizer ────────────────────
# Uses the same `histories` dict from the download cell above.
# Each bar = mean of final best_loss across seeds; error bar = std across seeds.

final_losses = {}
for opt_name, seed_hists in histories.items():
    finals = [h["best_loss"].iloc[-1] for h in seed_hists.values() if not h.empty]
    if finals:
        final_losses[opt_name] = finals

opts = sorted(final_losses.keys(), key=lambda o: np.mean(final_losses[o]))
means = [np.mean(final_losses[o]) for o in opts]
stds  = [np.std(final_losses[o]) for o in opts]

fig, ax = plt.subplots(figsize=(8, max(4, len(opts) * 0.45)))
colors = [optimizer_color(o) for o in opts]
ax.barh(opts, means, xerr=stds, capsize=4, color=colors, edgecolor="white")
ax.set_xscale("log")
ax.set_xlabel("Best Loss (log scale, lower is better)")
ax.set_title(f"Final Best Loss — model: {CHOSEN_MODEL.split('/')[-1]}, msg: {CHOSEN_MSG_ID}")
ax.invert_yaxis()

# Add lower-bound line if available
if lb_histories:
    lb_finals = [h["best_loss"].iloc[-1] for h in lb_histories.values() if not h.empty]
    if lb_finals:
        lb_mean = np.mean(lb_finals)
        ax.axvline(lb_mean, color="black", linestyle="--", linewidth=1.5, label="soft_prompt (lower bound)")
        ax.legend(fontsize=9)

plt.tight_layout()
fig.savefig(f"{RESULTS_DIR}/fig_bar_best_loss_m{CHOSEN_MSG_ID}.pdf", dpi=FIG_DPI, bbox_inches="tight")
plt.show()

# Load (Exp1 CSVs)

Loads CSV I/II/III from each per-model result file and concatenates them so
ranking / plots aggregate across all runs by default.

Comment out a line in `EXP1_MODELS` to exclude that model and inspect the
remaining ones in isolation.

In [ ]:
# ── Models to aggregate across ───────────────────────────────────────
# Each entry is the filename slug exp1.py used when writing its CSVs
# (i.e. `{RESULTS_DIR}/exp1_wb_{slug}_csv_{i,ii,iii}.csv`).
# Comment out any line to drop that model from the aggregated analysis.
# Missing CSVs are skipped with a warning rather than raising.
EXP1_MODELS = [
    "Llama-3.1-8B-Instruct",
    "gemma-3-12b-it",
    "Qwen3-8B",
    "gemma-4-26B-A4B-it",
]

# ── Optimizers to drop from every analysis below ─────────────────────
# Applied once after concat, so bar/box/Friedman/LaTeX all see the same set.
# Useful for excluding partial / still-running optimizers that would otherwise
# shrink the Friedman full-coverage task count. Soft optimizers are excluded
# separately, per-cell, via the `is_soft` column (not this list).
BLACKLIST_OPTIMIZERS: list[str] = [
    # "mac",
    "nanogcg",
    # "gaslite2",
]

def _load_if_exists(path: str) -> pd.DataFrame | None:
    if not os.path.exists(path):
        print(f"  [skip] missing {path}")
        return None
    return pd.read_csv(path)

frames_i, frames_ii, frames_iii = [], [], []
loaded_models = []
for slug in EXP1_MODELS:
    prefix = f"{RESULTS_DIR}/exp1_wb_{slug}"
    ci   = _load_if_exists(f"{prefix}_csv_i.csv")
    cii  = _load_if_exists(f"{prefix}_csv_ii.csv")
    ciii = _load_if_exists(f"{prefix}_csv_iii.csv")
    if ci is None and cii is None and ciii is None:
        continue
    loaded_models.append(slug)
    if ci   is not None: frames_i.append(ci)
    if cii  is not None: frames_ii.append(cii)
    if ciii is not None: frames_iii.append(ciii)

def _concat(frames):
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

csv_i   = _concat(frames_i)
csv_ii  = _concat(frames_ii)
csv_iii = _concat(frames_iii)

if BLACKLIST_OPTIMIZERS:
    before = {"i": len(csv_i), "ii": len(csv_ii), "iii": len(csv_iii)}
    for name, df in (("csv_i", csv_i), ("csv_ii", csv_ii), ("csv_iii", csv_iii)):
        if not df.empty and "optimizer_name" in df.columns:
            df.drop(df.index[df["optimizer_name"].isin(BLACKLIST_OPTIMIZERS)],
                    inplace=True)
    print(f"Blacklist {BLACKLIST_OPTIMIZERS} dropped: "
          f"csv_i {before['i']-len(csv_i)}, csv_ii {before['ii']-len(csv_ii)}, "
          f"csv_iii {before['iii']-len(csv_iii)} rows")

# ── Stable optimizer → color map (shared by every plot below) ────────
# Keyed alphabetically so rerunning with a different subset keeps each
# optimizer on the same color. Falls back to grey for unknown names.
_all_opts = sorted({
    o for df in (csv_i, csv_ii, csv_iii)
    if not df.empty and "optimizer_name" in df.columns
    for o in df["optimizer_name"].unique()
})
_palette = sns.color_palette(PALETTE, max(len(_all_opts), 20))
OPTIMIZER_COLORS: dict[str, tuple] = {o: _palette[i] for i, o in enumerate(_all_opts)}

def optimizer_color(name: str):
    return OPTIMIZER_COLORS.get(name, (0.5, 0.5, 0.5))

print(f"\nLoaded models ({len(loaded_models)}/{len(EXP1_MODELS)}): {loaded_models}")
print(f"CSV I   rows: {len(csv_i):>6}"
      + (f"  ({csv_i['model_name'].nunique()} models, "
         f"{csv_i['optimizer_name'].nunique()} optimizers)" if not csv_i.empty else ""))
print(f"CSV II  rows: {len(csv_ii):>6}")
print(f"CSV III rows: {len(csv_iii):>6}")
print(f"Optimizer palette: {len(OPTIMIZER_COLORS)} stable colors assigned")
if not csv_i.empty:
    print("Rows per model (CSV I):")
    print(csv_i.groupby("model_name").size().to_string())

## 2  Average Loss per Optimizer (CSV I)

Average final loss across all seeds and instructions.

In [ ]:
def bar_plot(df, value_col, title, ylabel, output_file=None,
             exclude_soft=True):
    """Bar plot: avg value per optimizer, with standard-error error bars.

    Aggregates across every row after filtering — so each row (one seed on one
    message on one model) is treated as an IID sample. SE therefore *over*states
    precision if you care about task-level uncertainty; prefer the box plot
    (section 5) for that.
    """
    plot_df = df.copy()
    if exclude_soft and "is_soft" in plot_df.columns:
        plot_df = plot_df[~plot_df["is_soft"].astype(bool)]

    agg = (plot_df.groupby("optimizer_name")[value_col]
           .agg(["mean", "std", "count"])
           .reset_index()
           .sort_values("mean"))
    agg["se"] = agg["std"] / np.sqrt(agg["count"])

    fig, ax = plt.subplots(figsize=(max(8, len(agg) * 0.7), 5))
    colors = [optimizer_color(o) for o in agg["optimizer_name"]]
    ax.bar(agg["optimizer_name"], agg["mean"], yerr=agg["se"],
           capsize=4, color=colors, edgecolor="white")
    ax.set_xlabel("Optimizer")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    plt.xticks(rotation=35, ha="right")
    ax.set_yscale("log")
    plt.tight_layout()
    if output_file:
        fig.savefig(output_file, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()
    return agg

agg_loss = bar_plot(csv_i, "best_loss", "Average Final Loss per Optimizer", "Loss (↓ better)",
                    output_file=f"{RESULTS_DIR}/fig_avg_loss.pdf")
agg_loss

## 2.5  Optimizer Ranking (Friedman + Performance Profiles)

Tasks = `(model, msg_id)` with seeds aggregated by mean. Reports:

- **Mean rank** per optimizer (Friedman-style; lower = better) + Friedman χ² test.
- **Critical Difference (CD)** at α=0.05 via Nemenyi post-hoc (Demšar 2006).
  Optimizers whose mean ranks differ by less than CD are **not** statistically
  distinguishable.
- **Performance profile** (Dolan–Moré): fraction of tasks where an optimizer is
  within a factor τ of the best optimizer on that task.

Soft optimizers are excluded from the ranking — they serve as a lower-bound
reference, not a competing method.


In [ ]:
from scipy import stats

METRIC = "best_loss"
LOWER_IS_BETTER = True

# Soft optimizers are excluded here (lower-bound reference, not a competitor).
# Global optimizer blacklist is already applied at load time.
df_rank = csv_i.copy()
if "is_soft" in df_rank.columns:
    df_rank = df_rank[~df_rank["is_soft"].astype(bool)]

# Aggregate seeds: one score per (optimizer, model, msg_id) task
task_scores = (df_rank.groupby(["optimizer_name", "model_name", "msg_id"])[METRIC]
               .mean().reset_index())

# Raw pivot: rows = tasks, cols = optimizers. Any NaN means an optimizer did
# not run on that task; dropna keeps only tasks with full coverage.
score_mat_raw = task_scores.pivot_table(
    index=["model_name", "msg_id"], columns="optimizer_name", values=METRIC,
)
score_mat = score_mat_raw.dropna(axis=0)
print(f"Tasks observed:       {len(score_mat_raw)}")
print(f"Tasks w/ full cover:  {len(score_mat)}  (dropped {len(score_mat_raw) - len(score_mat)})")
print(f"Optimizers compared:  {score_mat.shape[1]}")

if len(score_mat_raw) > 0:
    cov = (score_mat_raw.count() / len(score_mat_raw)).sort_values()
    partial = cov[cov < 1.0]
    if len(partial):
        print("\nOptimizers missing from some tasks "
              "(add to BLACKLIST_OPTIMIZERS in the load cell to recover tasks):")
        for opt, c in partial.items():
            print(f"  {opt:<20s} {c*100:5.1f}% ({int(round(c*len(score_mat_raw)))}/{len(score_mat_raw)})")

if len(score_mat) < 3 or score_mat.shape[1] < 3:
    print(f"\nSkipping Friedman / CD: need N≥3 tasks and k≥3 optimizers "
          f"(have N={len(score_mat)}, k={score_mat.shape[1]}).")
else:
    # Per-task ranks (1 = best); ties get average rank
    ranks = score_mat.rank(axis=1, ascending=LOWER_IS_BETTER, method="average")
    mean_ranks = ranks.mean(axis=0).sort_values()
    print("\nMean ranks (lower = better):")
    print(mean_ranks.to_string(float_format="%.2f"))

    # Friedman omnibus test
    stat, p = stats.friedmanchisquare(*[score_mat[c].values for c in score_mat.columns])
    print(f"\nFriedman chi^2 = {stat:.2f}, p = {p:.2e}")

    # Nemenyi critical difference at alpha=0.05 (Demsar 2006, Table 5)
    _Q_ALPHA_05 = {
        2: 1.960, 3: 2.343, 4: 2.569, 5: 2.728, 6: 2.850, 7: 2.949, 8: 3.031,
        9: 3.102, 10: 3.164, 11: 3.219, 12: 3.268, 13: 3.313, 14: 3.354,
        15: 3.391, 16: 3.426, 17: 3.458, 18: 3.489, 19: 3.517, 20: 3.544,
    }
    k = score_mat.shape[1]
    N = len(score_mat)
    q_alpha = _Q_ALPHA_05.get(k, 3.544)  # 20-optimizer value as a safe upper bound
    CD = q_alpha * np.sqrt(k * (k + 1) / (6 * N))
    print(f"\nCritical Difference (alpha=0.05, k={k}, N={N}): CD = {CD:.3f}")
    if N < 5:
        print(f"  [warn] N={N} is small — CD is loose and test is underpowered.")

    # Mean-rank bar with shaded "within-CD-of-best" region
    fig, ax = plt.subplots(figsize=(9, max(3.5, k * 0.32)))
    y_pos = np.arange(len(mean_ranks))
    ax.barh(y_pos, mean_ranks.values,
            color=[optimizer_color(o) for o in mean_ranks.index], edgecolor="white")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(mean_ranks.index)
    ax.invert_yaxis()
    best = mean_ranks.iloc[0]
    ax.axvspan(best, best + CD, color="grey", alpha=0.18,
               label=f"within CD of best (CD={CD:.2f})")
    ax.set_xlabel("Mean rank (1 = best)")
    ax.set_title(f"Friedman ranking - {METRIC} (Friedman p={p:.1e}, N={N})")
    ax.legend(loc="lower right", fontsize=9)
    plt.tight_layout()
    fig.savefig(f"{RESULTS_DIR}/fig_friedman_ranks.pdf", dpi=FIG_DPI, bbox_inches="tight")
    plt.show()


In [ ]:
# Dolan-More performance profile.
# r_{p,s} = score_p_s / best_p across optimizers s on problem p (>= 1).
# rho_s(tau) = fraction of problems where r_{p,s} <= tau.
# Requires the score_mat / mean_ranks built above.
TAU_MAX_CAP = 1e3  # losses span many orders of magnitude; cap tau for readability.

if "score_mat" not in globals() or len(score_mat) < 2 or score_mat.shape[1] < 2:
    print("Performance profile needs at least 2 tasks and 2 optimizers — skipped.")
else:
    mat = score_mat.values  # tasks x optimizers
    if LOWER_IS_BETTER:
        best_per_task = mat.min(axis=1, keepdims=True)
        ratios = mat / np.maximum(best_per_task, 1e-12)
    else:
        best_per_task = mat.max(axis=1, keepdims=True)
        ratios = best_per_task / np.maximum(mat, 1e-12)

    tau_max = min(max(float(np.nanmax(ratios)), 2.0), TAU_MAX_CAP)
    taus = np.logspace(0, np.log10(tau_max), 300)

    fig, ax = plt.subplots(figsize=(9, 5))
    # Iterate in mean-rank order so legend matches the ranking plot
    order = mean_ranks.index if "mean_ranks" in globals() else score_mat.columns
    for opt in order:
        j = list(score_mat.columns).index(opt)
        rho = (ratios[:, j][:, None] <= taus[None, :]).mean(axis=0)
        ax.plot(taus, rho, label=opt, color=optimizer_color(opt), linewidth=1.6)

    ax.set_xscale("log")
    ax.set_xlabel(r"$\tau$ (factor of best on each task)")
    ax.set_ylabel(r"$\rho_s(\tau)$ - fraction of tasks within $\tau\times$ best")
    ax.set_title(f"Performance profile (Dolan-More) - {METRIC}  "
                 f"(N={len(score_mat)}, k={score_mat.shape[1]})")
    ax.set_ylim(0, 1.02)
    ax.set_xlim(1, tau_max)
    ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)
    ax.grid(True, which="both", alpha=0.3)
    plt.tight_layout()
    fig.savefig(f"{RESULTS_DIR}/fig_performance_profile.pdf",
                dpi=FIG_DPI, bbox_inches="tight")
    plt.show()


In [ ]:
# Per-model rank heatmap. Same per-task ranks as the Friedman cell, but
# averaged *within* each model instead of globally — so specialists that win
# on one model and lose on others show up as uneven rows (watch for e.g.
# one green cell + two red cells on the same optimizer).
if "score_mat" not in globals() or len(score_mat) < 1 or score_mat.shape[1] < 2:
    print("Rank heatmap needs score_mat with tasks and >=2 optimizers - skipped.")
else:
    task_ranks = score_mat.rank(axis=1, ascending=LOWER_IS_BETTER, method="average")
    tasks_per_model = score_mat.groupby(level="model_name").size()

    # Rows = optimizers, cols = models, cell = mean rank on that model's tasks.
    heat = task_ranks.groupby(level="model_name").mean().T
    heat["_avg"] = heat.mean(axis=1)
    heat = heat.sort_values("_avg", ascending=True).drop(columns="_avg")
    pretty_cols = [c.split("/")[-1] for c in heat.columns]

    fig, ax = plt.subplots(figsize=(max(6, heat.shape[1] * 1.8),
                                     max(4, heat.shape[0] * 0.35)))
    sns.heatmap(heat.set_axis(pretty_cols, axis=1),
                annot=True, fmt=".1f", cmap="RdYlGn_r",
                cbar_kws={"label": "Mean rank (1 = best on that model)"},
                ax=ax, linewidths=0.5, linecolor="white")
    task_str = ", ".join(f"{m.split('/')[-1]}={n}" for m, n in tasks_per_model.items())
    ax.set_xlabel("Model")
    ax.set_ylabel("Optimizer")
    ax.set_title(f"Per-model mean rank - {METRIC}  (tasks per model: {task_str})")
    plt.tight_layout()
    fig.savefig(f"{RESULTS_DIR}/fig_rank_heatmap.pdf",
                dpi=FIG_DPI, bbox_inches="tight")
    plt.show()


## 3  Average BLEU per Optimizer (CSV II)

In [ ]:
agg_bleu = bar_plot(csv_ii, "bleu", "Average BLEU Score per Optimizer", "BLEU (↑ better)",
                    output_file=f"{RESULTS_DIR}/fig_avg_bleu.pdf")
agg_bleu


## 4  Average Universality per Optimizer (CSV III)

Universality = mean `strongreject_finetuned` score across ClearHarm messages.

In [ ]:
agg_univ = bar_plot(csv_iii, "strongreject_finetuned",
                    "Average Universality Score per Optimizer",
                    "Universality / Jailbreakness (↑ = more harmful)",
                    output_file=f"{RESULTS_DIR}/fig_avg_universality.pdf")
agg_univ


## 5  Box Plots

First average across seeds per (optimizer, model, msg_id) to reduce seed noise, then plot distribution across messages/models.

In [ ]:
# Filter control — set to None to include all models
FILTER_MODELS = None  # e.g. ["google/gemma-2-2b-it"]

def box_plot(df, value_col, title, ylabel, output_file=None,
             exclude_soft=True):
    plot_df = df.copy()
    if exclude_soft and "is_soft" in plot_df.columns:
        plot_df = plot_df[~plot_df["is_soft"].astype(bool)]
    if FILTER_MODELS:
        plot_df = plot_df[plot_df["model_name"].isin(FILTER_MODELS)]

    # Average across seeds first
    seed_avg = (plot_df.groupby(["optimizer_name", "model_name", "msg_id"])[value_col]
                .mean().reset_index())

    fig, ax = plt.subplots(figsize=(max(8, seed_avg["optimizer_name"].nunique() * 0.9), 5))
    order = (seed_avg.groupby("optimizer_name")[value_col].median()
             .sort_values().index.tolist())
    sns.boxplot(data=seed_avg, x="optimizer_name", y=value_col, order=order,
                palette={o: optimizer_color(o) for o in order}, ax=ax, width=0.6)
    ax.set_xlabel("Optimizer")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    if output_file:
        fig.savefig(output_file, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()

box_plot(csv_i,   "best_loss",              "Loss Distribution per Optimizer",         "Loss (↓ better)",
         f"{RESULTS_DIR}/fig_box_loss.pdf")
box_plot(csv_ii,  "bleu",                   "BLEU Distribution per Optimizer",          "BLEU (↑ better)",
         f"{RESULTS_DIR}/fig_box_bleu.pdf")
box_plot(csv_iii, "strongreject_finetuned", "Universality Distribution per Optimizer", "Universality (↑ = more harmful)",
         f"{RESULTS_DIR}/fig_box_universality.pdf")

## 6  LaTeX Table

Rows = optimizers, columns = models, cell = avg across seeds & instructions. Winners per column are marked in **bold**.

In [ ]:
METRIC_COL = "best_loss"  # swap to "bleu" or "strongreject_finetuned" for other metrics
METRIC_LOWER_IS_BETTER = True  # set False for BLEU / universality

df_tab = csv_i.copy()
if "is_soft" in df_tab.columns:
    df_tab = df_tab[~df_tab["is_soft"].astype(bool)]

# Average across seeds and msg_ids per (optimizer, model)
pivot = (df_tab.groupby(["optimizer_name", "model_name"])[METRIC_COL]
         .mean().unstack("model_name"))

# Add overall average column
pivot["Avg"] = pivot.mean(axis=1)
pivot = pivot.sort_values("Avg", ascending=METRIC_LOWER_IS_BETTER)

# Build LaTeX
model_cols = [c for c in pivot.columns if c != "Avg"]
header_row = " & ".join(["Optimizer"] + [c.split("/")[-1] for c in model_cols] + ["Avg"]) + r" \\"

rows_latex = []
for opt, row in pivot.iterrows():
    vals = [row[c] for c in model_cols] + [row["Avg"]]
    # Find winner (min or max) per column
    formatted = []
    for col_idx, (col, v) in enumerate(zip(model_cols + ["Avg"], vals)):
        col_vals = pivot[col].dropna()
        if pd.isna(v):
            formatted.append("—")
            continue
        is_winner = (v == col_vals.min()) if METRIC_LOWER_IS_BETTER else (v == col_vals.max())
        cell = f"\\textbf{{{v:.3f}}}" if is_winner else f"{v:.3f}"
        formatted.append(cell)
    rows_latex.append(f"    {opt} & " + " & ".join(formatted) + r" \\")

print(r"\begin{table}[h]")
print(r"\centering")
print(r"\caption{Optimizer Benchmark — " + METRIC_COL + "}")
print(r"\begin{tabular}{l" + "r" * (len(model_cols) + 1) + "}")
print(r"\toprule")
print(f"    {header_row}")
print(r"\midrule")
for r in rows_latex:
    print(r)
print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\end{table}")

---
# Exp2: Jailbreak Tweaks Analysis

Load exp2 CSVs (single and multi-instruction). Group by `variant_name` instead of `optimizer_name`.

**Controls**: Set `EXP2_RESULTS_DIR` and CSV filenames below to match your `run_all.sh` output.

In [ ]:
# ── Exp2 Configuration ──────────────────────────────────────────────
EXP2_RESULTS_DIR = RESULTS_DIR  # same dir by default

# exp2 is pinned to a single model (see run_all.sh EXP2_MODELS).
EXP2_MODEL_SLUG = "gemma-3-12b-it"

# Single-instruction CSVs
EXP2_SINGLE_CSV_I   = f"{EXP2_RESULTS_DIR}/exp2_single_{EXP2_MODEL_SLUG}_csv_i.csv"
EXP2_SINGLE_CSV_II  = f"{EXP2_RESULTS_DIR}/exp2_single_{EXP2_MODEL_SLUG}_csv_ii.csv"
EXP2_SINGLE_CSV_III = f"{EXP2_RESULTS_DIR}/exp2_single_{EXP2_MODEL_SLUG}_csv_iii.csv"

# Multi-instruction CSVs
EXP2_MULTI_CSV_I   = f"{EXP2_RESULTS_DIR}/exp2_multi_{EXP2_MODEL_SLUG}_csv_i.csv"
EXP2_MULTI_CSV_III = f"{EXP2_RESULTS_DIR}/exp2_multi_{EXP2_MODEL_SLUG}_csv_iii.csv"

# Load
import os
exp2s_i   = pd.read_csv(EXP2_SINGLE_CSV_I)   if os.path.exists(EXP2_SINGLE_CSV_I)   else pd.DataFrame()
exp2s_ii  = pd.read_csv(EXP2_SINGLE_CSV_II)  if os.path.exists(EXP2_SINGLE_CSV_II)  else pd.DataFrame()
exp2s_iii = pd.read_csv(EXP2_SINGLE_CSV_III) if os.path.exists(EXP2_SINGLE_CSV_III) else pd.DataFrame()
exp2m_i   = pd.read_csv(EXP2_MULTI_CSV_I)    if os.path.exists(EXP2_MULTI_CSV_I)    else pd.DataFrame()
exp2m_iii = pd.read_csv(EXP2_MULTI_CSV_III)   if os.path.exists(EXP2_MULTI_CSV_III)  else pd.DataFrame()

for name, df in [("exp2 single I", exp2s_i), ("exp2 single II", exp2s_ii),
                  ("exp2 single III", exp2s_iii), ("exp2 multi I", exp2m_i),
                  ("exp2 multi III", exp2m_iii)]:
    print(f"{name}: {len(df)} rows")

## Exp2 Single-Instruction: Loss, BLEU, Universality

Same plots as exp1 but grouped by `variant_name`.

In [ ]:
GROUP_COL = "variant_name"  # exp2 groups by variant, not optimizer

def bar_plot_exp2(df, value_col, title, ylabel, group_col=GROUP_COL, output_file=None):
    if df.empty:
        print(f"No data for: {title}")
        return
    agg = (df.groupby(group_col)[value_col]
           .agg(["mean", "std", "count"]).reset_index().sort_values("mean"))
    agg["se"] = agg["std"] / np.sqrt(agg["count"])
    fig, ax = plt.subplots(figsize=(max(8, len(agg) * 0.9), 5))
    colors = sns.color_palette(PALETTE, len(agg))
    ax.bar(agg[group_col], agg["mean"], yerr=agg["se"], capsize=4,
           color=colors, edgecolor="white")
    ax.set_xlabel("Variant")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    if output_file:
        fig.savefig(output_file, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()
    return agg

bar_plot_exp2(exp2s_i,   "best_loss",              "Exp2 Single: Avg Loss per Variant",         "Loss")
bar_plot_exp2(exp2s_ii,  "bleu",                   "Exp2 Single: Avg BLEU per Variant",         "BLEU")
bar_plot_exp2(exp2s_iii, "strongreject_finetuned", "Exp2 Single: Avg Universality per Variant", "Universality")

## Exp2 Multi-Instruction: Loss & Universality

Multi-instruction runs produce a single universal trigger per variant. 
CSV III universality is the primary metric here.

In [ ]:
bar_plot_exp2(exp2m_i,   "best_loss",              "Exp2 Multi: Avg Loss per Variant",         "Loss")
bar_plot_exp2(exp2m_iii, "strongreject_finetuned", "Exp2 Multi: Avg Universality per Variant", "Universality")

## Exp2 Box Plots (Single-Instruction)

Averaged across seeds, distribution across messages and models.

In [ ]:
def box_plot_exp2(df, value_col, title, ylabel, group_col=GROUP_COL, output_file=None):
    if df.empty:
        print(f"No data for: {title}")
        return
    seed_avg = (df.groupby([group_col, "model_name", "msg_id"])[value_col]
                .mean().reset_index())
    fig, ax = plt.subplots(figsize=(max(8, seed_avg[group_col].nunique() * 0.9), 5))
    order = (seed_avg.groupby(group_col)[value_col].median()
             .sort_values().index.tolist())
    sns.boxplot(data=seed_avg, x=group_col, y=value_col, order=order,
                palette=PALETTE, ax=ax, width=0.6)
    ax.set_xlabel("Variant")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    if output_file:
        fig.savefig(output_file, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()

box_plot_exp2(exp2s_i,   "best_loss",              "Exp2: Loss Distribution per Variant",         "Loss")
box_plot_exp2(exp2s_ii,  "bleu",                   "Exp2: BLEU Distribution per Variant",          "BLEU")
box_plot_exp2(exp2s_iii, "strongreject_finetuned", "Exp2: Universality Distribution per Variant", "Universality")

## Exp2 Optimization Dynamics (Single-Instruction)

Same as exp1 dynamics plot but filtered by `tweakbench_single` run type. Lines = variants.

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
EXP2_CHOSEN_MSG_ID = 0
EXP2_CHOSEN_MODEL  = exp2s_i["model_name"].iloc[0] if not exp2s_i.empty else "N/A"
EXP2_X_AXIS = "step"  # "step" or "flops"
EXP2_Y_AXIS = "best_loss"  # "loss" (per-step) or "best_loss" (running best)

print(f"Available msg_ids: {sorted(exp2s_i['msg_id'].dropna().unique().tolist()) if not exp2s_i.empty else []}")
print(f"Available models:  {exp2s_i['model_name'].unique().tolist() if not exp2s_i.empty else []}")

if not exp2s_i.empty:
    api = wandb.Api()
    runs = api.runs(
        f"{WANDB_ENTITY}/{WANDB_PROJECT}",
        filters={
            "state": "finished",
            "config.run_type": "tweakbench_single",
            "config.model_name": EXP2_CHOSEN_MODEL,
            "config.msg_id": EXP2_CHOSEN_MSG_ID,
        },
    )
    histories = {}
    for run in runs:
        var  = run.config.get("variant_name", "unknown")
        seed = run.config.get("seed", 0)
        try:
            hist = run.history(keys=["loss", "best_loss", "step", "total_models_stats/total_flops"], x_axis="_step", pandas=True)
        except Exception:
            hist = run.history(pandas=True)
        if hist.empty:
            continue
        if "step" not in hist.columns and "_step" in hist.columns:
            hist = hist.rename(columns={"_step": "step"})
        if "total_models_stats/total_flops" in hist.columns:
            hist = hist.rename(columns={"total_models_stats/total_flops": "flops"})
        histories.setdefault(var, {})[seed] = hist

    fig, ax = plt.subplots(figsize=(10, 5))
    colors = sns.color_palette(PALETTE, len(histories))
    for color, (var_name, seed_hists) in zip(colors, histories.items()):
        x_col = "flops" if (EXP2_X_AXIS == "flops" and "flops" in next(iter(seed_hists.values())).columns) else "step"
        all_x = sorted(set(x for h in seed_hists.values() for x in h[x_col].dropna()))
        interp_losses = []
        for hist in seed_hists.values():
            valid = hist[[x_col, EXP2_Y_AXIS]].dropna()
            if len(valid) < 2:
                continue
            interp_losses.append(np.interp(all_x, valid[x_col], valid[EXP2_Y_AXIS]))
        if not interp_losses:
            continue
        arr = np.array(interp_losses)
        mean, std = arr.mean(axis=0), arr.std(axis=0)
        ax.plot(all_x, mean, label=var_name, color=color)
        ax.fill_between(all_x, mean - std, mean + std, alpha=0.15, color=color)
    ax.set_xlabel("FLOPs" if EXP2_X_AXIS == "flops" else "Step")
    ax.set_ylabel(EXP2_Y_AXIS.replace("_", " ").title())
    ax.set_title(f"Exp2 Dynamics — {EXP2_CHOSEN_MODEL.split('/')[-1]}, msg {EXP2_CHOSEN_MSG_ID}")
    ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print("No exp2 single-instruction data available.")


## Exp2 LaTeX Table (Single-Instruction)

Rows = variants, columns = models, cell = avg across seeds & instructions.

In [ ]:
EXP2_METRIC_COL = "best_loss"  # swap to "bleu" or "strongreject_finetuned"
EXP2_METRIC_LOWER_IS_BETTER = True

if not exp2s_i.empty:
    pivot = (exp2s_i.groupby(["variant_name", "model_name"])[EXP2_METRIC_COL]
             .mean().unstack("model_name"))
    pivot["Avg"] = pivot.mean(axis=1)
    pivot = pivot.sort_values("Avg", ascending=EXP2_METRIC_LOWER_IS_BETTER)
    model_cols = [c for c in pivot.columns if c != "Avg"]
    header = " & ".join(["Variant"] + [c.split("/")[-1] for c in model_cols] + ["Avg"]) + r" \\"
    rows_latex = []
    for var, row in pivot.iterrows():
        vals = [row[c] for c in model_cols] + [row["Avg"]]
        formatted = []
        for col, v in zip(model_cols + ["Avg"], vals):
            col_vals = pivot[col].dropna()
            is_winner = (v == col_vals.min()) if EXP2_METRIC_LOWER_IS_BETTER else (v == col_vals.max())
            formatted.append(f"\\textbf{{{v:.3f}}}" if is_winner else f"{v:.3f}")
        rows_latex.append(f"    {var} & " + " & ".join(formatted) + r" \\")
    print(r"\begin{table}[h]")
    print(r"\centering")
    print(r"\caption{Jailbreak Tweaks — " + EXP2_METRIC_COL + "}")
    print(r"\begin{tabular}{l" + "r" * (len(model_cols) + 1) + "}")
    print(r"\toprule")
    print(f"    {header}")
    print(r"\midrule")
    for r in rows_latex:
        print(r)
    print(r"\bottomrule")
    print(r"\end{tabular}")
    print(r"\end{table}")
else:
    print("No exp2 data available.")